# 03 - Evaluation：模型評估與計分

本筆記本展示如何使用本框架的評估模組，對 LLM 工作流程進行系統性的品質量測。

## 簡介

評估管線（Evaluation Pipeline）是 LLM 開發流程中不可或缺的一環。本框架提供以下核心元件：

- **`EvaluationRunner`**：負責統籌執行評估流程，接受工作流函式、測試資料集與計分器，回傳彙整後的 `EvalResult`。
- **內建 Scorers**：
  - `ExactMatchScorer` — 判斷輸出是否與預期完全一致
  - `ContainsScorer` — 判斷輸出是否包含預期字串
  - `JsonValidScorer` — 判斷輸出是否為合法 JSON
- **`BaseScorer`**：抽象基底類別，可繼承並自訂計分邏輯。
- **`EvalResult`**：包含 `metrics`（彙整指標）、`details`（每筆詳細結果）、`run_id`（MLflow Run ID，若未啟用則為 `None`）。

整個流程支援 MLflow 追蹤，若 MLflow 服務不可用，會自動降級並仍回傳評估結果。

In [ ]:
import json
import os
import sys

# 將專案根目錄加入路徑（於 examples/ 目錄下執行時適用）
sys.path.insert(0, os.path.abspath('..'))

from app.evaluator import EvaluationRunner
from app.evaluator.scorers import (
    ExactMatchScorer,
    ContainsScorer,
    JsonValidScorer,
    BaseScorer,
)

print('匯入成功。')

## 準備測試資料

測試資料集（`test_cases`）是一個串列，每筆測試案例為一個字典，格式如下：

```python
{
    "input": {
        "system_prompt": "...",  # 系統提示詞
        "user_prompt": "...",    # 使用者輸入
    },
    "expected": "...",           # 預期輸出（字串）
    "metadata": {               # 可選的額外資訊
        "category": "...",
        "difficulty": "...",
    },
}
```

`metadata` 欄位可用於分組分析，例如依難度或類別查看各子集的表現。

In [ ]:
test_cases = [
    {
        "input": {
            "system_prompt": "你是一個簡潔的問答助理，請直接回答問題，不要加入多餘的說明。",
            "user_prompt": "台灣的首都是哪裡？",
        },
        "expected": "台北",
        "metadata": {"category": "地理", "difficulty": "easy"},
    },
    {
        "input": {
            "system_prompt": "你是一個 JSON 格式回應的助理，請以合法的 JSON 字串回答。",
            "user_prompt": "請以 JSON 格式回傳 name 為 Alice、age 為 30 的物件。",
        },
        "expected": '{"name": "Alice", "age": 30}',
        "metadata": {"category": "格式化", "difficulty": "medium"},
    },
    {
        "input": {
            "system_prompt": "你是一個程式碼說明助理，請用繁體中文解釋程式碼功能。",
            "user_prompt": "請說明 `list.append(x)` 的功能。",
        },
        "expected": "將元素 x 加入串列末尾",
        "metadata": {"category": "程式設計", "difficulty": "easy"},
    },
    {
        "input": {
            "system_prompt": "你是一個情緒分析助理，只能回答 positive、negative 或 neutral。",
            "user_prompt": "這個產品非常棒，我很滿意！",
        },
        "expected": "positive",
        "metadata": {"category": "NLP", "difficulty": "medium"},
    },
]

print(f'共準備 {len(test_cases)} 筆測試案例。')
for i, tc in enumerate(test_cases):
    print(f"  [{i}] 類別={tc['metadata']['category']}，預期輸出={repr(tc['expected'])}")

## 定義 Mock Workflow 函式

在實際使用中，`workflow_fn` 應為呼叫 LLM 並回傳字串回應的函式。
其簽名為：

```python
def workflow_fn(input: dict) -> str:
    ...
```

`input` 即測試案例中的 `"input"` 欄位，包含 `system_prompt` 與 `user_prompt`。

此處為了示範，使用預設回應的 Mock 函式，模擬模型的輸出。

In [ ]:
# Mock 工作流函式：根據 user_prompt 回傳預設的罐頭回應
MOCK_RESPONSES = {
    "台灣的首都是哪裡？": "台北",
    "請以 JSON 格式回傳 name 為 Alice、age 為 30 的物件。": '{"name": "Alice", "age": 30}',
    "請說明 `list.append(x)` 的功能。": "將元素 x 加入串列末尾，修改原始串列。",
    "這個產品非常棒，我很滿意！": "positive",
}


def mock_workflow_fn(input: dict) -> str:
    """模擬工作流函式，根據 user_prompt 回傳罐頭回應。"""
    user_prompt = input.get("user_prompt", "")
    response = MOCK_RESPONSES.get(user_prompt, "（無對應回應）")
    return response


# 快速測試 mock 函式
sample_input = test_cases[0]["input"]
sample_output = mock_workflow_fn(sample_input)
print(f'範例輸入：{sample_input["user_prompt"]}')
print(f'範例輸出：{sample_output}')

## 使用內建 Scorers

本框架提供三種內建計分器：

| Scorer | 計分邏輯 | 適用場景 |
|---|---|---|
| `ExactMatchScorer` | 輸出與預期完全相同得 1.0，否則得 0.0 | 分類、簡短問答 |
| `ContainsScorer` | 輸出包含預期字串得 1.0，否則得 0.0 | 關鍵字確認、部分比對 |
| `JsonValidScorer` | 輸出可成功解析為 JSON 得 1.0，否則得 0.0 | 結構化輸出驗證 |

`EvaluationRunner.evaluate()` 會對每筆測試案例套用所有計分器，
並在 `EvalResult.metrics` 中回傳每個計分器的平均分數。

In [ ]:
# 建立內建計分器
scorers = [
    ExactMatchScorer(),
    ContainsScorer(),
    JsonValidScorer(),
]

# 建立 EvaluationRunner 並執行評估
runner = EvaluationRunner()
result = runner.evaluate(
    workflow_fn=mock_workflow_fn,
    test_cases=test_cases,
    scorers=scorers,
    run_name="builtin_scorers_demo",
)

# 顯示彙整指標
print("=== 彙整指標（Metrics）===")
for metric_name, score in result.metrics.items():
    print(f"  {metric_name}: {score:.4f}")

if result.run_id:
    print(f"\nMLflow Run ID: {result.run_id}")
else:
    print("\n（MLflow 未啟用，結果僅保存於記憶體中）")

# 顯示每筆詳細結果
print("\n=== 每筆詳細結果（Details）===")
for i, detail in enumerate(result.details):
    print(f"\n[{i}] user_prompt: {detail['input']['user_prompt']}")
    print(f"    實際輸出:   {repr(detail['output'])}")
    print(f"    預期輸出:   {repr(detail['expected'])}")
    for scorer_name, scorer_result in detail['scores'].items():
        print(f"    {scorer_name}: {scorer_result['score']:.1f} — {scorer_result['reason']}")

## 自訂 Scorer

繼承 `BaseScorer` 並實作 `score(output, expected)` 方法，即可建立自訂計分器。

`score()` 必須回傳一個字典：

```python
{"score": float, "reason": str}
```

其中 `score` 範圍通常為 `[0.0, 1.0]`，`reason` 為說明字串，便於後續除錯分析。

以下示範一個 `LengthScorer`，根據輸出長度是否在合理範圍內給分：若輸出長度介於 1 到 50 字元之間則得滿分，否則按比例扣分。

In [ ]:
class LengthScorer(BaseScorer):
    """
    自訂計分器：根據輸出長度給分。
    - 輸出為空字串：得 0.0
    - 長度介於 min_len 至 max_len 之間：得 1.0
    - 超出上限：按比例遞減至最低 0.2
    """

    def __init__(self, min_len: int = 1, max_len: int = 50):
        self.min_len = min_len
        self.max_len = max_len

    def score(self, output: str, expected: str) -> dict:
        length = len(output)

        if length == 0:
            return {"score": 0.0, "reason": "輸出為空字串"}

        if self.min_len <= length <= self.max_len:
            return {
                "score": 1.0,
                "reason": f"長度 {length} 字元，符合範圍 [{self.min_len}, {self.max_len}]",
            }

        if length < self.min_len:
            ratio = length / self.min_len
            score_val = round(max(0.0, ratio), 4)
            return {
                "score": score_val,
                "reason": f"長度 {length} 字元，低於最小值 {self.min_len}，得分 {score_val}",
            }

        # length > max_len
        excess = length - self.max_len
        penalty = excess / self.max_len
        score_val = round(max(0.2, 1.0 - penalty), 4)
        return {
            "score": score_val,
            "reason": f"長度 {length} 字元，超出上限 {self.max_len}，得分 {score_val}",
        }


# 快速測試自訂計分器
length_scorer = LengthScorer(min_len=1, max_len=50)
print("自訂 LengthScorer 測試：")
for text in ["台北", "positive", '{"name": "Alice", "age": 30}', "將元素 x 加入串列末尾，修改原始串列。"]:
    r = length_scorer.score(text, "")
    print(f"  輸入={repr(text):40s} → 分數={r['score']:.2f}，原因={r['reason']}")

In [ ]:
# 使用自訂 LengthScorer 搭配 ExactMatchScorer 執行評估
custom_scorers = [
    ExactMatchScorer(),
    LengthScorer(min_len=1, max_len=50),
]

custom_result = runner.evaluate(
    workflow_fn=mock_workflow_fn,
    test_cases=test_cases,
    scorers=custom_scorers,
    run_name="custom_length_scorer_demo",
)

print("=== 含自訂 Scorer 的彙整指標 ===")
for metric_name, score in custom_result.metrics.items():
    print(f"  {metric_name}: {score:.4f}")

print("\n=== 每筆詳細結果 ===")
for i, detail in enumerate(custom_result.details):
    print(f"\n[{i}] 輸出={repr(detail['output'])}")
    for scorer_name, scorer_result in detail['scores'].items():
        print(f"    {scorer_name}: {scorer_result['score']:.4f} — {scorer_result['reason']}")

## 從檔案載入測試資料

在實際專案中，測試資料通常以 JSON 檔案形式版本控管，方便團隊共同維護與更新。
建議的 JSON 檔案格式如下：

```json
[
  {
    "input": {
      "system_prompt": "你是一個簡潔的問答助理。",
      "user_prompt": "台灣的首都是哪裡？"
    },
    "expected": "台北",
    "metadata": {
      "category": "地理",
      "difficulty": "easy"
    }
  }
]
```

使用 `json.dump()` 存檔、`json.load()` 讀檔，即可輕鬆整合至 CI/CD 流程，
在每次程式碼合併或模型更新後自動執行評估。

In [ ]:
import json
import os

# 將測試資料儲存為 JSON 檔案
test_data_path = os.path.join(os.path.abspath('..'), 'data', 'test_cases.json')
os.makedirs(os.path.dirname(test_data_path), exist_ok=True)

with open(test_data_path, 'w', encoding='utf-8') as f:
    json.dump(test_cases, f, ensure_ascii=False, indent=2)

print(f'測試資料已儲存至：{test_data_path}')

# 從 JSON 檔案重新載入測試資料
with open(test_data_path, 'r', encoding='utf-8') as f:
    loaded_test_cases = json.load(f)

print(f'成功載入 {len(loaded_test_cases)} 筆測試案例。')

# 使用載入的測試資料執行評估
file_result = runner.evaluate(
    workflow_fn=mock_workflow_fn,
    test_cases=loaded_test_cases,
    scorers=[ExactMatchScorer(), ContainsScorer()],
    run_name="from_file_demo",
)

print("\n=== 從檔案載入後的評估結果 ===")
for metric_name, score in file_result.metrics.items():
    print(f"  {metric_name}: {score:.4f}")

## 總結

本筆記本涵蓋了評估框架的完整使用流程：

1. **準備測試資料**：以標準字典格式定義 `input`、`expected` 與 `metadata`，便於分類分析。
2. **定義工作流函式**：`workflow_fn(input: dict) -> str` 是串接 LLM 邏輯與評估系統的橋樑。
3. **使用內建 Scorers**：`ExactMatchScorer`、`ContainsScorer`、`JsonValidScorer` 涵蓋大多數常見場景。
4. **自訂 Scorer**：繼承 `BaseScorer`，實作 `score(output, expected) -> dict` 即可無縫整合。
5. **從檔案載入**：以 JSON 格式版本控管測試資料，支援 CI/CD 自動化評估。

**後續建議**：
- 搭配 MLflow Tracking Server 保存每次評估歷史，比較不同模型版本的表現趨勢。
- 使用 `metadata` 中的 `category` 欄位進行子集分析，找出模型的弱點領域。
- 將評估步驟整合進 GitHub Actions 或其他 CI 工具，確保每次模型更新都有品質把關。